# Indonesian Batik Motifs Classification v3

This notebook implements the v3 architecture and training pipeline for the Indonesian Batik Motifs Detector.

### Major Architectural Fixes in v3:
1. **Double-Preprocessing Bug Resolved**: Fixed the dynamic range collapse where v2 normalized images twice ($[-1.008, -0.992]$). The model now has a single internal `Rescaling(scale=1/127.5, offset=-1.0)` layer and natively receives raw $[0, 255]$ float32 pixels.
2. **Perceptual Hash Deduplication**: Uses 64-bit dHash and Hamming distance to eliminate train/val/test data leakage from redundant duplicate images.
3. **Categorical Focal Loss & Effective Sample Weighting**: Replaces standard crossentropy with Focal Loss ($\gamma = 2.0$, label smoothing $= 0.1$) to prevent dominant classes from drowning out weak minority classes.
4. **Stratified 5-Fold Cross-Validation**: Two-phase fine-tuning (frozen backbone classification head training followed by top 40 layers unfreezing with frozen BatchNormalization).
5. **Zero-Backend Export**: Automatic serialization to FP16 TFLite and ONNX for client-side in-browser WebAssembly execution.

## 1. Environment Configuration and Hyperparameters


In [ ]:
"""Cell 1: Environment detection, deterministic seeding, and hyperparameters."""

import os
import random
import platform
import warnings
from pathlib import Path
import numpy as np
import tensorflow as tf

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Detect runtime environment paths
KAGGLE_DATASET = Path("/kaggle/input/datasets/fisheightcharacter/indonesian-batik-dataset-enhanced-and-cleaned/dataset")
LOCAL_DATASET = Path(r"D:\Personal Projects\Computer Vision-ML Projects\Batik Motifs Detector v2\Batik_dataset\dataset")
REPO_DATASET = Path("data/dataset")

if KAGGLE_DATASET.exists():
    DATASET_DIR = str(KAGGLE_DATASET)
    OUTPUT_DIR = "/kaggle/working"
elif LOCAL_DATASET.exists():
    DATASET_DIR = str(LOCAL_DATASET)
    OUTPUT_DIR = "./outputs/v3_train"
elif REPO_DATASET.exists():
    DATASET_DIR = str(REPO_DATASET)
    OUTPUT_DIR = "./outputs/v3_train"
else:
    DATASET_DIR = "./dataset"
    OUTPUT_DIR = "./outputs/v3_train"

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUTPUT_DIR, "best_batik_model_v3.keras")
TFLITE_PATH = os.path.join(OUTPUT_DIR, "batik_model_v3.tflite")
ONNX_PATH = os.path.join(OUTPUT_DIR, "batik_model_v3.onnx")
LABELS_PATH = os.path.join(OUTPUT_DIR, "labels.txt")
AUDIT_CSV_PATH = os.path.join(OUTPUT_DIR, "v3_dataset_audit.csv")

# Image input dimensions
IMG_HEIGHT, IMG_WIDTH = 224, 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
IMG_CHANNELS = 3
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)

# Training hyperparameters
BATCH_SIZE = 32
N_FOLDS = 5
TEST_SIZE = 0.15
PHASE1_EPOCHS = 20
PHASE2_EPOCHS = 40
PHASE1_LR = 1e-3
PHASE2_LR = 1e-5
PHASE1_PATIENCE = 5
PHASE2_PATIENCE = 7
REDUCE_LR_PATIENCE = 3
REDUCE_LR_FACTOR = 0.3
MIN_LR = 1e-7

# Model architecture hyperparameters
UNFREEZE_LAYERS = 40
DROPOUT_RATE_1 = 0.5
DROPOUT_RATE_2 = 0.3
DENSE_UNITS = 256
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.1
EFFECTIVE_BETA = 0.999
TTA_STEPS = 5

TIER1_THRESHOLD = 45
TIER2_THRESHOLD = 90

# GPU memory growth and mixed precision configuration
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU devices detected: {len(gpus)}")
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print(f"Mixed precision policy set to: {tf.keras.mixed_precision.global_policy().name}")
else:
    print("WARNING: No GPU detected. Execution will proceed on CPU.")

print("\nRuntime Configuration:")
print(f"  Platform         : {platform.platform()}")
print(f"  Python           : {platform.python_version()}")
print(f"  TensorFlow       : {tf.__version__}")
print(f"  Backbone         : EfficientNetV2S (v3 fixed pipeline)")
print(f"  Dataset directory: {DATASET_DIR}")
print(f"  Output directory : {OUTPUT_DIR}")
print(f"  Image dimensions : {IMG_SIZE}")
print(f"  Batch size       : {BATCH_SIZE}")
print(f"  Stratified Folds : {N_FOLDS}")
print(f"  Focal loss gamma : {FOCAL_GAMMA}")
print(f"  Label smoothing  : {LABEL_SMOOTHING}")


## 2. Dataset Discovery, Deduplication, and Class Profiling


In [ ]:
"""Cell 2: Dataset discovery, integrity checks, pHash deduplication, and class profiling."""

import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import LabelEncoder


def compute_dhash_int(image_path: str, hash_size: int = 8) -> int:
    """Computes difference hash (dHash) as a 64-bit integer."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
            pixels = np.array(img, dtype=np.int32)
            diff = pixels[:, 1:] > pixels[:, :-1]
            decimal_val = 0
            for bit in diff.flatten():
                decimal_val = (decimal_val << 1) | int(bit)
            return decimal_val
    except Exception:
        return 0


def hamming_distance(h1: int, h2: int) -> int:
    """Computes bitwise Hamming distance between two integers."""
    return bin(h1 ^ h2).count("1")


def discover_dataset(root_dir: str) -> pd.DataFrame:
    """Discovers all valid images grouped by class subdirectory."""
    valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    records = []
    root_path = Path(root_dir)

    if not root_path.exists():
        raise FileNotFoundError(f"Dataset directory not found: {root_dir}")

    class_dirs = sorted([d for d in root_path.iterdir() if d.is_dir()])
    if not class_dirs:
        raise ValueError(f"No class subdirectories found in: {root_dir}")

    for class_dir in class_dirs:
        label = class_dir.name
        for filepath in class_dir.iterdir():
            if filepath.is_file() and filepath.suffix.lower() in valid_extensions:
                records.append({"filepath": str(filepath), "label": label})

    return pd.DataFrame(records)


df = discover_dataset(DATASET_DIR)
print(f"Total raw images discovered: {len(df)}")

# Filter corrupted images and sub-resolution files
corrupted, dimension_issues = [], []
for idx, row in df.iterrows():
    img = cv2.imread(row["filepath"])
    if img is None:
        corrupted.append(idx)
        continue
    h, w = img.shape[:2]
    if h < 32 or w < 32:
        dimension_issues.append(idx)

if corrupted:
    df.drop(index=corrupted, inplace=True)
if dimension_issues:
    df.drop(index=dimension_issues, inplace=True)

df.reset_index(drop=True, inplace=True)
print(f"Valid images after integrity verification: {len(df)}")
print(f"  Corrupted files removed : {len(corrupted)}")
print(f"  Sub-resolution removed  : {len(dimension_issues)}")

# Deduplication via Perceptual Hash
print("\nExecuting perceptual hash deduplication scan...")
hashes = [compute_dhash_int(fp) for fp in df["filepath"]]
df["dhash"] = hashes

# Check if audit file exists in results/ or execute local duplicate detection
duplicate_indices = set()
for i in range(len(df)):
    if i in duplicate_indices:
        continue
    h1 = df.iloc[i]["dhash"]
    c1 = df.iloc[i]["label"]
    for j in range(i + 1, len(df)):
        if j in duplicate_indices:
            continue
        h2 = df.iloc[j]["dhash"]
        if hamming_distance(h1, h2) <= 2:
            duplicate_indices.add(j)

if duplicate_indices:
    print(f"Removing {len(duplicate_indices)} duplicate/near-duplicate images to prevent leakage...")
    df.drop(index=list(duplicate_indices), inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"Deduplicated dataset pool: {len(df)} images")

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["label"])
CLASS_NAMES = list(label_encoder.classes_)
NUM_CLASSES = len(CLASS_NAMES)

with open(LABELS_PATH, "w") as f:
    f.write("\n".join(CLASS_NAMES))

print(f"Saved class labels ({NUM_CLASSES} classes) to: {LABELS_PATH}")

# Class distribution analysis
class_counts = (
    df.groupby("label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

class_counts["tier"] = class_counts["count"].apply(
    lambda x: "Tier 1 (Aggressive)" if x <= TIER1_THRESHOLD
    else ("Tier 2 (Moderate)" if x <= TIER2_THRESHOLD else "Tier 3 (Light)")
)

tier_colors = {
    "Tier 1 (Aggressive)": "#d62728",
    "Tier 2 (Moderate)":   "#ff7f0e",
    "Tier 3 (Light)":      "#2ca02c",
}

fig, ax = plt.subplots(figsize=(14, 12))
sorted_counts = class_counts.sort_values("count", ascending=True)
bar_colors = [tier_colors[t] for t in sorted_counts["tier"]]
bars = ax.barh(sorted_counts["label"], sorted_counts["count"], color=bar_colors, edgecolor="white", height=0.75)
for bar, count in zip(bars, sorted_counts["count"]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, str(count), va="center", fontsize=8)

ax.axvline(x=TIER1_THRESHOLD, color="#d62728", linestyle="--", linewidth=1.2, label=f"Tier 1 Threshold ({TIER1_THRESHOLD})")
ax.axvline(x=TIER2_THRESHOLD, color="#ff7f0e", linestyle="--", linewidth=1.2, label=f"Tier 2 Threshold ({TIER2_THRESHOLD})")
ax.set_title("Class Distribution and Augmentation Tier Profile (Batik v3)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "v3_class_distribution.png"), dpi=150)
plt.close()


## 3. Stratified Holdout, Tier-Based Augmentations, and Data Pipelines


In [ ]:
"""Cell 3: Stratified train/test holdout, Albumentations tier pipelines, and tf.data loader."""

import cv2
import numpy as np
import albumentations as A
import tensorflow as tf
from sklearn.model_selection import train_test_split, StratifiedKFold

# Fixed 15% test holdout (never touched during K-Fold training or validation)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    df["filepath"].values,
    df["label_encoded"].values,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["label_encoded"].values,
)

print(f"Fixed test holdout pool : {len(X_test)} images")
print(f"Train/Validation pool   : {len(X_trainval)} images")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Tier-based Albumentations pipelines
AUGMENT_TIER1 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=30, p=0.6),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.4),
    A.ElasticTransform(alpha=80, sigma=8, p=0.3),
    A.RandomResizedCrop(size=IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(16, 32), hole_width_range=(16, 32), p=0.3),
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.3),
    A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.2), p=0.2),
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_TIER2 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=20, p=0.4),
    A.RandomResizedCrop(size=IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1), p=0.4),
    A.GaussianBlur(blur_limit=(3, 3), p=0.2),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_TIER3 = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_VALIDATION = A.Compose([
    A.Resize(height=IMG_HEIGHT, width=IMG_WIDTH),
])

AUGMENT_TTA = AUGMENT_TIER3


def get_tier_for_class(class_name: str) -> str:
    count = df[df["label"] == class_name].shape[0]
    if count <= TIER1_THRESHOLD:
        return "tier1"
    elif count <= TIER2_THRESHOLD:
        return "tier2"
    return "tier3"


CLASS_TIER_MAP = {i: get_tier_for_class(CLASS_NAMES[i]) for i in range(NUM_CLASSES)}


def load_image(filepath: str) -> np.ndarray:
    """Loads image from disk in RGB format. Returns raw float32 pixels in [0, 255]."""
    img = cv2.imread(filepath)
    if img is None:
        img = np.zeros((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8)
    img_rgb = cv2.cvtColor(cv2.resize(img, IMG_SIZE), cv2.COLOR_BGR2RGB)
    return img_rgb.astype(np.float32)


def augment_train(filepath: bytes, label: int) -> tuple:
    img = load_image(filepath.numpy().decode("utf-8"))
    label_int = int(label.numpy())
    tier = CLASS_TIER_MAP.get(label_int, "tier2")
    if tier == "tier1":
        img = AUGMENT_TIER1(image=img)["image"]
    elif tier == "tier2":
        img = AUGMENT_TIER2(image=img)["image"]
    else:
        img = AUGMENT_TIER3(image=img)["image"]
    return img.astype(np.float32), label_int


def augment_eval(filepath: bytes, label: int) -> tuple:
    img = load_image(filepath.numpy().decode("utf-8"))
    img = AUGMENT_VALIDATION(image=img)["image"]
    return img.astype(np.float32), int(label.numpy())


def augment_tta(filepath: bytes, label: int) -> tuple:
    img = load_image(filepath.numpy().decode("utf-8"))
    img = AUGMENT_TTA(image=img)["image"]
    return img.astype(np.float32), int(label.numpy())


def _wrap(aug_fn):
    """Wraps Python numpy augmentation function for tf.data graph execution.

    CRITICAL FIX: Note that preprocess_input is intentionally OMITTED here.
    The model's internal Rescaling layer directly processes raw [0, 255] float32 pixels.
    """
    def tf_fn(filepath, label):
        img, lbl = tf.py_function(func=aug_fn, inp=[filepath, label], Tout=[tf.float32, tf.int32])
        img.set_shape([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
        lbl.set_shape([])
        return img, tf.one_hot(lbl, depth=NUM_CLASSES)
    return tf_fn


tf_augment_train = _wrap(augment_train)
tf_augment_eval = _wrap(augment_eval)
tf_augment_tta = _wrap(augment_tta)

AUTOTUNE = tf.data.AUTOTUNE


def build_dataset(filepaths, labels, augment_fn, batch_size, shuffle=False, cache=False):
    """Constructs optimized tf.data.Dataset pipeline."""
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(filepaths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    return ds.batch(batch_size).prefetch(AUTOTUNE)


test_ds = build_dataset(X_test, y_test, tf_augment_eval, BATCH_SIZE, shuffle=False, cache=True)

print("Augmentation tier summary:")
for tier_name, tier_key in [("Tier 1 (Aggressive)", "tier1"), ("Tier 2 (Moderate)", "tier2"), ("Tier 3 (Light)", "tier3")]:
    classes_in_tier = [CLASS_NAMES[i] for i, t in CLASS_TIER_MAP.items() if t == tier_key]
    print(f"  {tier_name}: {len(classes_in_tier)} classes")


## 4. Model Architecture, Explicit Rescaling, and Focal Loss Formulation


In [ ]:
"""Cell 4: V3 Backbone model architecture, Categorical Focal Loss, and effective sample weighting."""

from typing import Sequence, Any
import numpy as np
import tensorflow as tf


@tf.keras.utils.register_keras_serializable(package="batik_v3")
class CategoricalFocalLoss(tf.keras.losses.Loss):
    """Categorical Focal Loss with label smoothing support.

    Formulation:
        FL(p_t) = - alpha_t * (1 - p_t)^gamma * log(p_t)
    """

    def __init__(
        self,
        gamma: float = 2.0,
        alpha: Any = None,
        label_smoothing: float = 0.1,
        reduction: str = "sum_over_batch_size",
        name: str = "categorical_focal_loss",
    ):
        super().__init__(reduction=reduction, name=name)
        self.gamma = float(gamma)
        self.alpha = alpha
        self.label_smoothing = float(label_smoothing)

    def call(self, y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)

        if self.label_smoothing > 0.0:
            num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        modulating_factor = tf.pow(1.0 - y_pred, self.gamma)
        log_prob = tf.math.log(y_pred)

        focal_cross_entropy = -y_true * modulating_factor * log_prob

        if self.alpha is not None:
            alpha_tensor = tf.cast(self.alpha, tf.float32)
            focal_cross_entropy = alpha_tensor * focal_cross_entropy

        return tf.reduce_sum(focal_cross_entropy, axis=-1)

    def get_config(self) -> dict[str, Any]:
        config = super().get_config()
        config.update({
            "gamma": self.gamma,
            "alpha": self.alpha,
            "label_smoothing": self.label_smoothing,
        })
        return config


def compute_effective_sample_weights(
    class_counts: Sequence[int],
    beta: float = 0.999,
) -> np.ndarray:
    """Computes class weights based on Effective Number of Samples formulation.

    W_c = (1 - beta) / (1 - beta^{N_c})
    """
    counts = np.array(class_counts, dtype=np.float64)
    effective_num = 1.0 - np.power(beta, counts)
    effective_num = np.where(effective_num <= 0.0, 1e-8, effective_num)

    weights = (1.0 - beta) / effective_num
    normalized_weights = weights / np.sum(weights) * len(counts)
    return normalized_weights.astype(np.float32)


def create_v3_backbone_model(
    num_classes: int = 28,
    input_shape: tuple[int, int, int] = (224, 224, 3),
    weights: str | None = "imagenet",
) -> tuple[tf.keras.Model, tf.keras.Model]:
    """Builds the v3 EfficientNetV2S classifier with single explicit Rescaling layer.

    Accepts raw [0, 255] float32 pixels. EfficientNetV2S has include_preprocessing=False.
    """
    inputs = tf.keras.Input(shape=input_shape, name="input_image", dtype=tf.float32)

    # Explicit single rescaling: [0, 255] -> [-1, 1]
    rescaled = tf.keras.layers.Rescaling(
        scale=1.0 / 127.5,
        offset=-1.0,
        name="input_rescaling",
    )(inputs)

    base_model = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        weights=weights,
        include_preprocessing=False,
        input_shape=input_shape,
    )

    features = base_model(rescaled)

    # Classification head
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(features)
    x = tf.keras.layers.BatchNormalization(name="head_bn")(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE_1, name="head_dropout_1")(x)
    x = tf.keras.layers.Dense(DENSE_UNITS, activation="relu", name="head_dense")(x)
    x = tf.keras.layers.Dropout(DROPOUT_RATE_2, name="head_dropout_2")(x)
    outputs = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="batik_efficientnetv2s_v3")
    return base_model, model


print("Model architecture and Focal Loss modules ready.")


## 5. Stratified 5-Fold Training Loop (Phase 1 & Phase 2 Fine-Tuning)


In [ ]:
"""Cell 5: Stratified 5-Fold cross-validation training loop with two-phase fine-tuning."""

import gc
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import TopKCategoricalAccuracy

fold_records = []
best_overall_val_acc = 0.0
best_overall_model_path = MODEL_PATH

for fold, (train_idx, val_idx) in enumerate(skf.split(X_trainval, y_trainval)):
    print(f"\n{'='*65}")
    print(f"  STARTING STRATIFIED FOLD {fold + 1} / {N_FOLDS}")
    print(f"{'='*65}")

    X_tr, X_val = X_trainval[train_idx], X_trainval[val_idx]
    y_tr, y_val = y_trainval[train_idx], y_trainval[val_idx]

    # Compute effective sample weights for current fold
    tr_counts = [np.sum(y_tr == c) for c in range(NUM_CLASSES)]
    class_weights_arr = compute_effective_sample_weights(tr_counts, beta=EFFECTIVE_BETA)
    fold_class_weights = dict(enumerate(class_weights_arr))

    train_ds_fold = build_dataset(X_tr, y_tr, tf_augment_train, BATCH_SIZE, shuffle=True)
    val_ds_fold = build_dataset(X_val, y_val, tf_augment_eval, BATCH_SIZE, shuffle=False)

    base_model, model = create_v3_backbone_model(num_classes=NUM_CLASSES, input_shape=INPUT_SHAPE, weights="imagenet")

    # Phase 1: Train classification head with frozen backbone
    base_model.trainable = False

    loss_fn = CategoricalFocalLoss(gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)

    model.compile(
        optimizer=Adam(learning_rate=PHASE1_LR),
        loss=loss_fn,
        metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )

    print(f"  [Fold {fold + 1}] Phase 1: Training classification head ({PHASE1_EPOCHS} epochs max)...")
    history_p1 = model.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=PHASE1_EPOCHS,
        callbacks=[
            EarlyStopping(monitor="val_accuracy", patience=PHASE1_PATIENCE, restore_best_weights=True, verbose=0, mode="max"),
            ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=0),
        ],
        class_weight=fold_class_weights,
        verbose=1,
    )

    # Phase 2: Unfreeze top N layers; keep all BatchNorm layers frozen
    base_model.trainable = True
    for layer in base_model.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in base_model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False

    fold_ckpt_path = os.path.join(OUTPUT_DIR, f"fold_{fold + 1}_best.keras")

    model.compile(
        optimizer=Adam(learning_rate=PHASE2_LR),
        loss=loss_fn,
        metrics=["accuracy", TopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )

    print(f"  [Fold {fold + 1}] Phase 2: Fine-tuning last {UNFREEZE_LAYERS} layers ({PHASE2_EPOCHS} epochs max)...")
    history_p2 = model.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=PHASE2_EPOCHS,
        callbacks=[
            ModelCheckpoint(filepath=fold_ckpt_path, monitor="val_accuracy", save_best_only=True, mode="max", verbose=0),
            EarlyStopping(monitor="val_accuracy", patience=PHASE2_PATIENCE, restore_best_weights=True, verbose=0, mode="max"),
            ReduceLROnPlateau(monitor="val_loss", factor=REDUCE_LR_FACTOR, patience=REDUCE_LR_PATIENCE, min_lr=MIN_LR, verbose=0),
        ],
        class_weight=fold_class_weights,
        verbose=1,
    )

    best_val_acc = max(history_p2.history["val_accuracy"])
    best_val_top3 = max(history_p2.history["val_top3_accuracy"])

    if best_val_acc > best_overall_val_acc:
        best_overall_val_acc = best_val_acc
        model.save(best_overall_model_path)
        print(f"  --> New overall best model saved: val_acc = {best_val_acc:.4f}")

    fold_records.append({
        "fold": fold + 1,
        "val_accuracy": float(best_val_acc),
        "val_top3_accuracy": float(best_val_top3),
    })
    print(f"  Fold {fold + 1} Results -> Val Acc: {best_val_acc:.4f} | Val Top-3: {best_val_top3:.4f}")

    del model, base_model, train_ds_fold, val_ds_fold
    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "=" * 65)
print("  5-FOLD CROSS-VALIDATION SUMMARY")
print("=" * 65)
for rec in fold_records:
    print(f"  Fold {rec['fold']} -> Top-1: {rec['val_accuracy']:.4f} | Top-3: {rec['val_top3_accuracy']:.4f}")
mean_acc = np.mean([r["val_accuracy"] for r in fold_records])
print(f"\nMean Validation Accuracy across {N_FOLDS} folds: {mean_acc:.4f}")


## 6. Holdout Evaluation, Test-Time Augmentation (TTA), and Classification Report


In [ ]:
"""Cell 6: Test holdout evaluation, Test-Time Augmentation (TTA), and comprehensive metrics."""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

print("Loading best model for test holdout evaluation...")
best_model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={"CategoricalFocalLoss": CategoricalFocalLoss},
)

# Standard evaluation on clean test set
print("Evaluating on clean test holdout...")
clean_eval = best_model.evaluate(test_ds, verbose=1)
print(f"Clean Test Holdout Loss    : {clean_eval[0]:.4f}")
print(f"Clean Test Holdout Accuracy: {clean_eval[1]:.4f}")

# Test-Time Augmentation (TTA)
print(f"\nExecuting {TTA_STEPS}-step Test-Time Augmentation (TTA)...")
tta_predictions = []

# Step 0: Clean pass
clean_preds = best_model.predict(test_ds, verbose=0)
tta_predictions.append(clean_preds)

# Steps 1..TTA_STEPS: Lightly augmented passes
for step in range(TTA_STEPS):
    tta_ds = build_dataset(X_test, y_test, tf_augment_tta, BATCH_SIZE, shuffle=False)
    preds = best_model.predict(tta_ds, verbose=0)
    tta_predictions.append(preds)

# Ensemble average
ensemble_probs = np.mean(tta_predictions, axis=0)
y_pred_classes = np.argmax(ensemble_probs, axis=1)

# Overall Accuracy Metrics
top1_acc = np.mean(y_pred_classes == y_test)
top3_hits = [y_test[i] in np.argsort(ensemble_probs[i])[-3:] for i in range(len(y_test))]
top3_acc = np.mean(top3_hits)

print("\n" + "=" * 65)
print("  FINAL TEST HOLDOUT METRICS (WITH TTA)")
print("=" * 65)
print(f"  Top-1 Accuracy : {top1_acc * 100:.2f}%")
print(f"  Top-3 Accuracy : {top3_acc * 100:.2f}%")

# Classification Report
target_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
clf_report = classification_report(y_test, y_pred_classes, target_names=target_names, digits=4)
print("\nPer-Class Classification Report:")
print(clf_report)

# Save report to disk
report_path = os.path.join(OUTPUT_DIR, "v3_classification_report.txt")
with open(report_path, "w") as f:
    f.write(clf_report)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(18, 16))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.title("Confusion Matrix (V3 Test Holdout with TTA)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, "v3_confusion_matrix.png")
plt.savefig(cm_path, dpi=150)
plt.close()
print(f"Saved confusion matrix visualization to: {cm_path}")


## 7. Model Serialization (TFLite & ONNX FP16) and Parity Verification


In [ ]:
"""Cell 7: Model export to TFLite and ONNX (FP16), followed by numerical parity validation."""

import os
import subprocess
import numpy as np
import tensorflow as tf

print("Step 1: Exporting FP16 TFLite model...")
converter = tf.lite.TFLiteConverter.from_saved_model(MODEL_PATH) if os.path.isdir(MODEL_PATH) else None
if converter is None:
    best_model = tf.keras.models.load_model(
        MODEL_PATH,
        custom_objects={"CategoricalFocalLoss": CategoricalFocalLoss},
    )
    converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
print(f"Exported TFLite model: {TFLITE_PATH} ({tflite_size_mb:.2f} MB)")

print("\nStep 2: Exporting ONNX model via tf2onnx...")
onnx_cmd = [
    "python", "-m", "tf2onnx.convert",
    "--keras", MODEL_PATH,
    "--output", ONNX_PATH,
    "--opset", "17",
]
try:
    subprocess.run(onnx_cmd, check=True)
    onnx_size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
    print(f"Exported ONNX model: {ONNX_PATH} ({onnx_size_mb:.2f} MB)")
except Exception as e:
    print(f"ONNX conversion warning: {e}. If running locally, install tf2onnx (`pip install tf2onnx`).")

print("\nStep 3: Numerical Parity Verification across runtimes...")
dummy_input = np.random.uniform(0.0, 255.0, size=(1, 224, 224, 3)).astype(np.float32)

# Keras prediction
keras_pred = best_model(dummy_input, training=False).numpy()[0]

# TFLite prediction
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]["index"], dummy_input)
interpreter.invoke()
tflite_pred = interpreter.get_tensor(output_details[0]["index"])[0]

max_diff_tflite = np.max(np.abs(keras_pred - tflite_pred))
print(f"  Max absolute deviation (Keras vs TFLite FP16): {max_diff_tflite:.6f}")
assert max_diff_tflite < 1e-2, f"Discrepancy too high: {max_diff_tflite}"

# ONNX prediction (if onnxruntime installed)
if os.path.exists(ONNX_PATH):
    try:
        import onnxruntime as ort
        ort_sess = ort.InferenceSession(ONNX_PATH)
        onnx_input_name = ort_sess.get_inputs()[0].name
        onnx_pred = ort_sess.run(None, {onnx_input_name: dummy_input})[0][0]

        max_diff_onnx = np.max(np.abs(keras_pred - onnx_pred))
        print(f"  Max absolute deviation (Keras vs ONNX): {max_diff_onnx:.6f}")
        assert max_diff_onnx < 1e-3, f"ONNX Discrepancy too high: {max_diff_onnx}"
    except ImportError:
        print("  onnxruntime not installed in this environment; skipping ONNX runtime verification.")

print("\nModel serialization and parity verification complete.")
